<a href="https://colab.research.google.com/github/bsheese/387/blob/main/20_how_networks_learn/20_1_Network_Anatomy/20_1_1_Anatomy_of_a_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Networks Learn: Part 3
## Building a Network

---

## What This Notebook Is About

You now have all the pieces of a neural network, built separately:

- **19_1_3**: a neuron is a weighted sum of its inputs plus a bias, passed through an activation function, and a layer of neurons is a matrix multiplication.
- **19_1_5**: one neuron can't solve XOR, but a hidden layer of two neurons can. We chose the hidden weights by hand.
- **20_0**: autograd gives the slope of the loss for every weight, and gradient descent uses the slopes to improve the weights.

This notebook puts them together with PyTorch's building blocks, and lets gradient descent find the hidden layer's weights on its own.

**What you will learn:**
1. `nn.Linear`, PyTorch's layer of neurons, and how it matches the weighted sums from 19_1_3
2. What a Python class is, and `nn.Module`, the pattern every PyTorch model follows
3. Why the hidden layer uses the ReLU activation instead of 19_1_5's step function
4. How a network with a hidden layer learns XOR, how to count its weights and biases, and what its hidden layer computes
5. `nn.Sequential`, a shorter way to write a model whose layers run in order, and `state_dict()`, the weights PyTorch saves to a file

**Optional reference.** If you took CS 377, you have seen these ideas before in [18_2_3](https://github.com/bsheese/cs377/blob/main/18_classification/18_2_LogisticRegression/18_2_3_LogReg_to_NeuralNetworks.ipynb). If you haven't, that notebook covers the same ideas on another dataset, in more depth. You don't need it for this course.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])    # XOR: 1 when exactly one input is 1
print(X.shape, y.shape)

These are the same four XOR points as in 19_1_5. `y` has shape (4, 1), one column, because the network will give one output per point.

---

## Section 1: `nn.Linear`, a Layer of Neurons

`nn.Linear(n_inputs, n_outputs)` is a layer of `n_outputs` neurons, each with one weight per input and a bias. It creates the weights for you (starting at small random values) and computes the weighted sums. You've already used one: the last layer of the Pets model in 19_0 was `nn.Linear(512, 4)`.

In [ ]:
torch.manual_seed(0)
layer = nn.Linear(2, 3)            # 3 neurons, each with 2 weights and a bias
print('weight:', layer.weight.shape)
print('bias:  ', layer.bias.shape)

The weight matrix has one row per neuron, shape (3, 2), the same layout as the Pets model's (4, 512) matrix in 19_1_3. Calling the layer on the four points computes `X @ weight.T + bias`, the matrix multiplication from 19_1_3:

In [ ]:
out = layer(X)
by_hand = X @ layer.weight.T + layer.bias
print(out.shape)
print('same as by hand:', torch.allclose(out, by_hand))

The output has shape (4, 3): one row per point and one column per neuron. Its weights were created with `requires_grad=True`, so autograd tracks everything computed from them.

---

## Section 2: `nn.Module`, the Shape of Every Model

A PyTorch model is written as a Python **class**, so this section starts with what a class is. A class bundles some data together with the functions that work on that data. You have already used objects made from classes: the `layer` from Section 1 holds its own data (`layer.weight` and `layer.bias`), and the optimizer in 20_0_2 holds the weights it updates and has a function, `optimizer.step()`, that changes them. Here is a small class that has nothing to do with neural networks:

In [ ]:
class Counter:
    def __init__(self, start):
        self.count = start              # store a value on the object

    def add(self, n):
        self.count = self.count + n     # change the stored value
        return self.count

clicks = Counter(10)     # make one Counter object; this runs __init__ with start = 10
print(clicks.add(1))
print(clicks.add(5))
print(clicks.count)

`Counter(10)` makes one object from the class, and making an object runs its `__init__` function automatically, with 10 as `start`. Inside the class, **`self`** is the object itself. `self.count = start` stores a value on the object, and every function in the class gets `self` as its first input, so `add` can read and change `self.count`. In `clicks.add(1)` Python fills in `self` for you, and you only pass `n`. The printout shows 11, then 16, and then 16 again, because the count stays stored on `clicks` between calls.

A PyTorch model is a class built on `nn.Module`. Writing `class OneNeuron(nn.Module):` means the new class starts with everything `nn.Module` already does and adds its own layers. It has two parts:

- **`__init__`** creates the layers and stores them on `self`, the way `Counter` stored its count. Its first line is always `super().__init__()`, which runs `nn.Module`'s own `__init__` so that the setup PyTorch needs is done before any layers are stored. PyTorch then finds every layer stored on `self` and collects its weights, so `model.parameters()` can hand them all to an optimizer.
- **`forward(x)`** says how an input becomes an output. You don't call it directly. `nn.Module` is written so that calling `model(x)` runs `forward(x)`, the same way calling `layer(X)` in Section 1 computed the layer's weighted sums.

Here's the single neuron from 19_1_3 (logistic regression) as a module: one `nn.Linear` with one output, followed by sigmoid.

In [ ]:
class OneNeuron(nn.Module):
    def __init__(self):
        super().__init__()                 # always the first line: sets up the nn.Module machinery
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

torch.manual_seed(0)
neuron = OneNeuron()
print(neuron)
print('probabilities before training:', neuron(X).detach().squeeze().numpy().round(3))

### Training it on XOR

The loss for a yes/no problem is **binary cross-entropy**, `nn.BCELoss`. It's the cross-entropy loss from 19_1_4 for two classes: for each point, $-\log$ of the probability the model gave to the right answer, averaged over the points. The training loop is the one from 20_0_2, with `torch.optim.SGD` taking the steps.

In [ ]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(neuron.parameters(), lr=0.5)

for step in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(neuron(X), y)
    loss.backward()
    optimizer.step()

print(f'final loss: {loss.item():.4f}   (-log(0.5) = {-np.log(0.5):.4f})')
print('probabilities after training:', neuron(X).detach().squeeze().numpy().round(3))

The same result as scikit-learn's logistic regression in 19_1_5: after 2,000 steps the neuron gives every point a probability of 0.5, and the loss is $-\log(0.5)$. A single neuron draws one straight line, no straight line separates XOR, and "50/50 everywhere" is the best it can do by the loss.

---

## Section 3: The Hidden Layer Needs an Activation It Can Learn With

In 19_1_5 the hidden neurons used a **step** activation: 1 if the weighted sum is above 0, otherwise 0. It made the arithmetic easy, but gradient descent can't learn with it. Use the nudge test from 20_0_1 on a step function:

In [ ]:
def step(z):
    return (z > 0).float()

def nudge_slope(f, z, h=0.001):
    return (f(z + h) - f(z - h)) / (2 * h)

for z in [-2.0, -0.5, 0.5, 2.0]:
    print(f'slope of step at z = {z:4}: {nudge_slope(step, torch.tensor(z)).item()}')

The slope is 0 everywhere (except exactly at 0, where the step jumps). Because rates multiply (20_0_1), a slope of 0 anywhere in the chain makes the slope of the loss 0 for every weight before it. With step activations, every hidden weight would have a slope of 0, and gradient descent would never move them.

The activation used in most networks today is **ReLU** (rectified linear unit): $\text{ReLU}(z) = \max(0, z)$. It passes positive values through unchanged and turns negative values into 0.

In [ ]:
zs = torch.linspace(-3, 3, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(zs, step(zs))
ax1.set_title('step: slope 0 everywhere')
ax2.plot(zs, torch.relu(zs))
ax2.set_title('ReLU: slope 1 for positive z')
for ax in (ax1, ax2):
    ax.set_xlabel('weighted sum z')
plt.tight_layout()
plt.show()

for z in [-2.0, -0.5, 0.5, 2.0]:
    print(f'slope of ReLU at z = {z:4}: {nudge_slope(torch.relu, torch.tensor(z)).item():.2f}')

ReLU's slope is 1 wherever the weighted sum is positive, so slopes can pass back through it to the weights before it. It's still bent, not a straight line, and that bend is what lets a hidden layer compute new features that a straight line couldn't. (If the hidden layer had no activation at all, two layers of weighted sums would just add up to one weighted sum, and the network would be a single straight line again.)

---

## Section 4: A Network That Learns XOR

The network below has a hidden layer of 4 ReLU neurons and an output layer of one sigmoid neuron:

In [ ]:
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)    # 2 inputs -> 4 hidden neurons
        self.output = nn.Linear(4, 1)    # 4 hidden values -> 1 output

    def forward(self, x):
        h = torch.relu(self.hidden(x))           # the hidden layer's 4 new features
        return torch.sigmoid(self.output(h))     # one neuron on top, as in 19_1_3

for name, p in XORNet().named_parameters():
    print(f'{name:14} shape {str(list(p.shape)):8} {p.numel():3} numbers')
print('total:', sum(p.numel() for p in XORNet().parameters()))

`named_parameters()` gives each of a model's weight tensors with its name (`hidden.weight`, `hidden.bias` and so on), and `.numel()` counts the numbers in one tensor. Section 5 comes back to listing a model's weights.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.axis('off')
ax.set_aspect('equal')
positions = {'input': (0.5, [2.0, 1.0]), 'hidden': (2.5, [3.0, 2.0, 1.0, 0.0]), 'output': (4.5, [1.5])}
names = {'input': ['x1', 'x2'], 'hidden': ['h1', 'h2', 'h3', 'h4'], 'output': ['p']}
for (a, b) in [('input', 'hidden'), ('hidden', 'output')]:
    for ya in positions[a][1]:
        for yb in positions[b][1]:
            ax.plot([positions[a][0], positions[b][0]], [ya, yb], color='#BBB', lw=0.8, zorder=1)
for layer_name, (x_pos, ys) in positions.items():
    for y_pos, label in zip(ys, names[layer_name]):
        ax.add_patch(plt.Circle((x_pos, y_pos), 0.22, color='#D6EAF8', ec='#333', zorder=2))
        ax.text(x_pos, y_pos, label, ha='center', va='center', zorder=3)
ax.text(0.5, 3.7, 'inputs', ha='center', fontweight='bold')
ax.text(2.5, 3.7, 'hidden layer (ReLU)', ha='center', fontweight='bold')
ax.text(4.5, 3.7, 'output (sigmoid)', ha='center', fontweight='bold')
ax.text(1.5, -0.9, 'nn.Linear(2, 4)\n8 weights + 4 biases', ha='center', fontsize=9)
ax.text(3.5, -0.9, 'nn.Linear(4, 1)\n4 weights + 1 bias', ha='center', fontsize=9)
ax.set_xlim(-0.2, 5.2)
ax.set_ylim(-1.3, 4.0)
plt.show()

Every line in the diagram is one weight. The hidden layer's weight matrix has shape (4, 2), one row per hidden neuron, and the output layer's has shape (1, 4).

**Counting a layer's numbers.** In `nn.Linear(n_in, n_out)`, each of the `n_out` neurons has one weight for each of the `n_in` inputs, so the layer has `n_in × n_out` weights, plus `n_out` biases, one per neuron. For XORNet's hidden layer, `nn.Linear(2, 4)`, that is 2 × 4 = 8 weights and 4 biases, 12 numbers. For the output layer, `nn.Linear(4, 1)`, it is 4 × 1 = 4 weights and 1 bias, 5 numbers. Adding up the layers gives 12 + 5 = 17, the total printed above. The same procedure gives the 2,052 numbers of the Pets model's last layer in 19_0_1: 512 × 4 = 2,048 weights plus 4 biases. The next cell does the count in code:

In [ ]:
def count_linear(n_in, n_out):
    return n_in * n_out + n_out      # weights + biases

print('hidden layer:', count_linear(2, 4))
print('output layer:', count_linear(4, 1))
print('total:       ', count_linear(2, 4) + count_linear(4, 1))

The count matches the printout from `named_parameters()`, where the hidden layer's 8 + 4 and the output layer's 4 + 1 add up to 17. These 17 numbers are what gradient descent has to set. Training uses exactly the same loop as the single neuron:

In [ ]:
def train(model, steps=2000, lr=0.5):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(42)
net = XORNet()
losses = train(net)

plt.figure(figsize=(7, 3.5))
plt.plot(losses)
plt.xlabel('step')
plt.ylabel('loss (binary cross-entropy)')
plt.show()

with torch.no_grad():
    p = net(X)
for xi, yi, pi in zip(X, y, p):
    print(f'x = {xi.tolist()}   target {int(yi.item())}   probability {pi.item():.3f}')

The loss falls to nearly 0, and the network gives each point a probability very close to its target. Nobody chose the hidden weights; gradient descent found them, starting from random values.

Here is the network's answer for every point in the square around the four XOR points. The shading shows where the network says 1 (red) or 0 (blue), and the dashed curve is where its probability is exactly 0.5.

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = torch.tensor(np.column_stack([grid_x.ravel(), grid_y.ravel()]), dtype=torch.float32)
with torch.no_grad():
    probs = net(grid).numpy().reshape(grid_x.shape)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.contourf(grid_x, grid_y, probs, levels=[0, 0.5, 1], colors=['#D6EAF8', '#FADBD8'])
ax.contour(grid_x, grid_y, probs, levels=[0.5], colors='k', linestyles='--')
for xi, yi in zip(X, y):
    ax.scatter(xi[0], xi[1], s=250, c='#C0392B' if yi.item() == 1 else '#2980B9', edgecolors='black', zorder=3)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title("XORNet's decision boundary")
plt.show()

The boundary isn't a single straight line. It's made of two lines that form a band: the two blue corners fall inside the band and the two red corners fall outside it. No single neuron can draw a boundary like that, because each neuron contributes one straight line, and here the hidden neurons' lines have been combined.

### What the hidden layer computes

As in 19_1_5, the hidden layer turns each point's two inputs into new features, here four of them. These are the values the output neuron actually sees:

In [ ]:
with torch.no_grad():
    hidden_values = torch.relu(net.hidden(X))
print('  x1 x2  |  h1     h2     h3     h4    | target')
for xi, hi, yi in zip(X, hidden_values, y):
    print(f'  {int(xi[0])}  {int(xi[1])}   | ' + '  '.join(f'{v:5.2f}' for v in hi) + f'  |   {int(yi.item())}')
print('output weights:', net.output.weight.detach().numpy().round(2), ' bias:', net.output.bias.detach().numpy().round(2))

The learned features are harder to read than 19_1_5's "at least one" and "both", and they'll differ each time the network is trained from a different start. They do the same job, though. The output neuron is still one weighted sum followed by sigmoid, and with these four features its weighted sum is high for the two target-1 points and low for the other two. The hidden layer has rearranged the points so that a single neuron's straight boundary works, and it learned to do that from the loss alone.

---

## Section 5: `nn.Sequential`, and Listing a Model's Weights

When a model just runs its layers one after another, `nn.Sequential` saves writing a class. This is the same network as `XORNet`:

In [ ]:
seq = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid(),
)
print(seq)

(`nn.ReLU()` and `nn.Sigmoid()` are the layer versions of `torch.relu` and `torch.sigmoid`, so they can go in the list.) Use `nn.Sequential` for straight stacks like this, and write an `nn.Module` class when the forward pass needs anything more, such as using an input twice or combining two branches.

Every model can list its weights by name with `named_parameters()`, and `state_dict()` returns all of them as a dictionary of tensors. A state dictionary is what `torch.save` writes to a file. The `pets4_head.pt` file from 19_0 was the state dictionary of one `nn.Linear(512, 4)`, which is why it held a `weight` and a `bias`.

In [ ]:
for name, tensor in seq.state_dict().items():
    print(f'{name:10} {list(tensor.shape)}')

The four names are `0.weight`, `0.bias`, `2.weight` and `2.bias`. `nn.Sequential` names each layer by its position in the list, and the ReLU (position 1) and the sigmoid (position 3) have no weights, so they don't appear. The shapes, [4, 2], [4], [1, 4] and [1], are the same as XORNet's.

---

## Summary

- `nn.Linear(n_inputs, n_outputs)` is a layer of neurons: a weight matrix of shape (n_outputs, n_inputs) and one bias per neuron. It computes `x @ weight.T + bias`, the matrix multiplication from 19_1_3.
- A **class** bundles data and the functions that use it, and `self` is the object itself. A model is a class built on `nn.Module`: `__init__` runs when the model is made, calls `super().__init__()` first and creates the layers, and `forward` says how they're used. Calling `model(x)` runs `forward`, and `model.parameters()` gives every weight to the optimizer.
- A single neuron trained on XOR ends at a probability of 0.5 everywhere, as in 19_1_5.
- The step activation has a slope of 0, so gradient descent can't train weights before it. **ReLU**, $\max(0, z)$, has a slope of 1 for positive inputs and is still bent, which is what a hidden layer needs.
- `nn.Linear(n_in, n_out)` has `n_in × n_out` weights and `n_out` biases. XORNet (2 inputs, 4 hidden ReLU neurons, 1 sigmoid output) has 8 + 4 + 4 + 1 = 17 numbers, and it learns XOR with **binary cross-entropy** and gradient descent. Its hidden layer learns features that make the points separable for the output neuron.
- `nn.Sequential` stacks layers in order. `named_parameters()` and `state_dict()` list a model's weights, and a state dictionary is what `torch.save` writes to a file, such as `pets4_head.pt` in 19_0.

## What's next

XORNet learned from four points, and every training step used all four. 21_0_1 builds the full training loop on real data: Fashion-MNIST, small grayscale images of clothing and shoes, with 10,000 training images in 10 classes. Each image is turned into a row of 784 numbers, and the model is an `nn.Sequential` like the one in Section 5, with a hidden layer of ReLU neurons. With that much data the loop needs pieces this notebook didn't: the training images are handed to the model in shuffled batches, the loss for 10 classes is the cross-entropy loss from 19_1_4 instead of the yes/no version used here, and the model is checked on validation images while it trains.